# ASX Multi-Factor Stock Screener

A single screener that scans either the **ASX 200**, a **named watchlist**, or a
**single stock**, and scores every company across four independent lenses:

| Sheet | Lens | What it checks |
|---|---|---|
| `Graham` | Classic value baseline | Graham Number, P/E, P/B, current ratio, margin of safety |
| `Momentum` | Price trend | Price vs 200-day SMA, 6M / 12M returns |
| `Cash_Generation` | Cash quality + DCF | FCF margin, cash conversion, shareholder yield, a basic 5-year DCF intrinsic value |
| `Leverage` | Balance-sheet risk | Debt/Equity, Net Debt/EBITDA, interest coverage, current/quick ratio, net debt trend |

Each lens produces its own **0-6 score**; `Summary` sums them into a single
**Composite Score (0-24)** and `Candidates` lists everything that clears the
minimum bar on all four lenses at once (see `min_*_score` in the config cell).

`Charts` plots normalised price performance against the **ASX 200 (^AXJO)** and
**S&P 500 (^GSPC)** baselines, plus a few cross-sectional scatter/bar charts.

## Universe modes (set `universe_mode` in the next cell)
- `'asx200'` - scrapes the current S&P/ASX 200 constituent list from Wikipedia at
  run time (index membership changes quarterly, so this is never frozen/stale).
  **Slow**: ~200 tickers x several API calls each, expect well over 10 minutes.
- `'watchlist'` - runs one of the named lists in the `watchlists` dict below.
- `'single'` - a single-ticker deep dive.

## Before you run
- yfinance's free fundamentals feed is patchy, especially for ASX small/mid-caps.
  Every ratio here degrades to `NaN` (never a crash) when a line item is missing,
  and the `Data_Issues` sheet + `Fundamentals Source` column tell you why. Use the
  optional debug cell at the bottom to inspect the raw statement row labels for
  any ticker showing unexpected blanks.
- The DCF and CAPM-style assumptions (discount rate, growth caps, score
  thresholds) are point-in-time inputs on the `Assumptions` sheet - review them,
  don't just trust the defaults.

In [ ]:
from pathlib import Path
from datetime import datetime
import io
import time
import warnings

import numpy as np
import pandas as pd
import requests
import yfinance as yf

# --- Universe ---------------------------------------------------------------
# 'asx200'    -> scrape the current S&P/ASX 200 list from Wikipedia (slow, wide)
# 'watchlist' -> run one of the named lists in `watchlists` below
# 'single'    -> one ticker, deep dive
universe_mode = 'watchlist'

watchlists = {
    # Originally carried over from the old liquidity-screen notebook, plus
    # electrical / data-centre construction contractors (SXE onwards).
    'Construction': [
        'SHA.AX', 'SYL.AX', 'VYS.AX', 'IPG.AX',
        'SKS.AX', 'APX.AX', 'HUB.AX', 'OCL.AX', 'TLS.AX',
        'SXE.AX',   # Southern Cross Electrical Engineering - data centre electrical fit-out
        'GNP.AX',   # GenusPlus - power/electrical infrastructure
        'MND.AX',   # Monadelphous - electrical & instrumentation, energy infrastructure
        'VNT.AX',   # Ventia - infrastructure services, telco/data networks
        'DOW.AX',   # Downer - engineering, power & utilities construction
    ],
    # Carried over from the old cash-generation notebook's sector example.
    'ASX Telecommunications': ['TLS.AX', 'TPG.AX', 'CDA.AX'],
    'Big Four Banks': ['CBA.AX', 'NAB.AX', 'WBC.AX', 'ANZ.AX'],
    # ASX-listed banks (ADIs) and non-bank lenders. The ASX has nowhere near 50
    # listed lenders, so this is close to the full set rather than a top 50.
    # NOTE: Graham current ratio / Leverage / FCF metrics are built for
    # industrial companies and read poorly for lenders (deposits and loan books
    # are their "debt" and "capex") - judge these mainly on P/E, P/B, ROE and
    # momentum.
    'Banking': [
        # Banks
        'CBA.AX', 'WBC.AX', 'NAB.AX', 'ANZ.AX', 'MQG.AX', 'BOQ.AX', 'BEN.AX',
        'JDO.AX', 'MYS.AX', 'AMP.AX', 'HGH.AX',
        # Non-bank lenders: mortgages / consumer / BNPL
        'PPM.AX', 'LFG.AX', 'RMC.AX', 'AFG.AX', 'MME.AX', 'PLT.AX', 'WZR.AX',
        'HMY.AX', 'HUM.AX', 'ZIP.AX', 'B4P.AX', 'PYR.AX',
        # Non-bank lenders: auto / equipment / novated lease / SME / specialty
        'SVR.AX', 'CCV.AX', 'COG.AX', 'MMS.AX', 'FPR.AX', 'EPY.AX', 'BTN.AX',
        'QFE.AX', 'MAF.AX',
        # Credit managers / debt purchasers
        'CCP.AX', 'PNC.AX',
    ],
    # Energy generators/retailers/infrastructure, weighted to renewables and to
    # names supplying (or owning) data-centre capacity. MEZ/MCY/CEN/GNE are NZ
    # renewable generators that are also listed on the ASX.
    'Green Energy': [
        'AGL.AX',   # AGL - largest generator, big renewables/battery build-out
        'ORG.AX',   # Origin - Eraring transition, batteries, Octopus stake
        'APA.AX',   # APA - gas pipelines plus electricity transmission/renewables
        'IFT.AX',   # Infratil - owns CDC Data Centres and Longroad renewables
        'MEZ.AX',   # Meridian - 100% renewable (hydro/wind)
        'MCY.AX',   # Mercury - hydro/geothermal/wind
        'CEN.AX',   # Contact - geothermal/hydro
        'GNE.AX',   # Genesis - hydro plus thermal transition
        'LPE.AX',   # Locality Planning Energy - embedded networks / solar
    ],
}
active_watchlist = 'Construction'   # used when universe_mode == 'watchlist'

single_ticker = 'TLS.AX'            # used when universe_mode == 'single'

# --- Benchmarks ---------------------------------------------------------------
# yfinance uses an UPPERCASE ".AX" suffix for ASX-listed stocks (e.g. "BHP.AX") -
# a lowercase ".xa" suffix, as sometimes shown on broker sites, returns nothing.
asx200_ticker = '^AXJO'    # S&P/ASX 200 index
sp500_ticker = '^GSPC'     # S&P 500 index
# NOTE: the price comparison charts below index everything to a base of 100 and
# compare % growth trajectories - they do NOT currency-adjust the S&P 500 (USD)
# against the ASX names (AUD). That's fine for comparing trend/shape, but do not
# read the two indices' lines as directly convertible dollar returns.
price_period = '5y'

# Only the top N tickers by Composite Score (plus the two benchmarks) go on the
# normalised-price line chart - with 200 ASX-wide tickers a full legend would be
# unreadable. Every ticker's numbers still land in the data sheets regardless.
chart_max_tickers = 15

# --- Graham (value baseline) score thresholds --------------------------------
graham_pe_max = 15.0
graham_pb_max = 1.5
graham_current_ratio_min = 2.0
graham_debt_to_equity_max = 1.0

# --- Cash Generation score thresholds -----------------------------------------
min_fcf_margin = 0.10          # Free Cash Flow / Revenue
min_cash_conversion = 0.80     # Operating Cash Flow / Net Income
max_capex_intensity = 0.15     # |CapEx| / Revenue
min_shareholder_yield = 0.02   # (Dividends + Buybacks) / Market Cap

# --- Basic DCF assumptions ----------------------------------------------------
dcf_discount_rate = 0.09       # required return / WACC proxy
dcf_terminal_growth = 0.025    # perpetuity growth after the projection window
dcf_projection_years = 5
# Historical FCF CAGR drives the projection, but is clamped to this range so a
# single volatile year can't produce an absurd valuation.
dcf_growth_floor = -0.05
dcf_growth_cap = 0.15
dcf_growth_fallback = 0.03     # used when FCF history is too short/negative to trust a CAGR

# --- Leverage score thresholds (this notebook's main focus area) -------------
leverage_debt_to_equity_max = 1.0
leverage_net_debt_to_ebitda_max = 3.0
leverage_interest_coverage_min = 4.0
leverage_current_ratio_min = 1.2
leverage_quick_ratio_min = 1.0

# --- Composite "Candidates" screen: minimum score (0-6) required per lens ----
min_graham_score = 3
min_momentum_score = 2
min_cashgen_score = 3
min_leverage_score = 4

# Yahoo Finance will start returning empty/partial data if hit with too many
# rapid requests. A short pause between tickers makes the download far more
# reliable, especially for a full ASX 200 scan.
request_pause_seconds = 0.6

# --- Output location -----------------------------------------------------------
preferred_output_folder = Path(r'C:\Users\dougm\OneDrive\Documents\Trading\2026.08')
try:
    preferred_output_folder.mkdir(parents=True, exist_ok=True)
    output_folder = preferred_output_folder
except OSError as exc:
    fallback_folder = Path.home() / 'ASX_Stock_Screener'
    fallback_folder.mkdir(parents=True, exist_ok=True)
    warnings.warn(
        f"Could not create/access '{preferred_output_folder}' ({exc}). "
        f"Writing output to '{fallback_folder}' instead."
    )
    output_folder = fallback_folder

if universe_mode == 'watchlist':
    report_tag = active_watchlist.replace(' ', '_')
elif universe_mode == 'single':
    report_tag = single_ticker.replace('.AX', '')
elif universe_mode == 'asx200':
    report_tag = 'ASX200'
else:
    raise ValueError(f"Unknown universe_mode '{universe_mode}' - use 'asx200', 'watchlist' or 'single'.")

output_file = output_folder / f'ASX_Stock_Screener_{report_tag}.xlsx'
print(f'Universe mode: {universe_mode} | Output will be written to: {output_file}')

In [ ]:
def safe_get(statement, labels):
    '''Pull the most recent value for the first matching row label in a
    yfinance financial statement DataFrame. Returns NaN if the statement is
    missing/empty or none of the labels are present (different companies
    report under slightly different line-item names).'''
    if statement is None or statement.empty:
        return np.nan
    for label in labels:
        if label in statement.index:
            value = statement.loc[label]
            if isinstance(value, pd.Series):
                return pd.to_numeric(value.iloc[0], errors='coerce')
            return pd.to_numeric(value, errors='coerce')
    return np.nan


def get_row_series(statement, labels):
    '''Like safe_get, but returns the FULL multi-period row (every column
    yfinance returned, typically up to 4 annual periods) sorted chronologically
    ascending, instead of just the latest value. Needed for trend charts and
    FCF CAGR - safe_get alone can't see history.'''
    if statement is None or statement.empty:
        return pd.Series(dtype=float)
    for label in labels:
        if label in statement.index:
            row = pd.to_numeric(statement.loc[label], errors='coerce')
            return row.sort_index()
    return pd.Series(dtype=float)


def safe_info(ticker_obj):
    '''company.info can raise (rate limiting, delisted tickers, network
    issues) or return None instead of an empty dict, so always route through
    this wrapper rather than calling .info directly.'''
    try:
        info = ticker_obj.info
        return info if info else {}
    except Exception:
        return {}


def get_financials(company, annual_attr, quarterly_attr):
    '''Yahoo Finance's free fundamentals feed is patchy for ASX small/mid-caps:
    the annual statement is often empty even when a quarterly one exists, and
    for some tickers neither is available at all. Prefer annual, fall back to
    quarterly, and report which one (if either) was actually used so a blank
    ratio can be told apart from a genuinely weak one.'''
    def _safe(attr):
        try:
            df = getattr(company, attr)
            return df if df is not None else pd.DataFrame()
        except Exception:
            return pd.DataFrame()

    annual = _safe(annual_attr)
    if not annual.empty:
        return annual, 'annual'
    quarterly = _safe(quarterly_attr)
    if not quarterly.empty:
        return quarterly, 'quarterly'
    return pd.DataFrame(), 'unavailable'


def calc_cagr(start_value, end_value, years):
    if pd.isna(start_value) or pd.isna(end_value) or start_value <= 0 or years <= 0:
        return np.nan
    return (end_value / start_value) ** (1 / years) - 1


def excel_column_name(col_num):
    '''0-indexed column number -> Excel column letter(s), e.g. 0 -> 'A'.'''
    name = ''
    col_num += 1
    while col_num:
        col_num, remainder = divmod(col_num - 1, 26)
        name = chr(65 + remainder) + name
    return name


def graham_score(eps, pe, pb, current_ratio, debt_to_equity, margin_of_safety):
    '''0-6 classic Graham value checklist: real earnings, cheap relative to
    earnings and book value, conservatively financed, and trading below the
    Graham Number (i.e. a positive margin of safety).'''
    checks = [
        pd.notna(eps) and eps > 0,
        pd.notna(pe) and pe > 0 and pe < graham_pe_max,
        pd.notna(pb) and pb > 0 and pb < graham_pb_max,
        pd.notna(current_ratio) and current_ratio > graham_current_ratio_min,
        pd.notna(debt_to_equity) and debt_to_equity < graham_debt_to_equity_max,
        pd.notna(margin_of_safety) and margin_of_safety > 0,
    ]
    return int(sum(checks))


def momentum_score(pct_vs_sma200, return_6m, return_12m):
    '''0-3: above its 200-day SMA (primary trend up), and positive 6M / 12M
    returns (trend has follow-through, not just a single good week).'''
    checks = [
        pd.notna(pct_vs_sma200) and pct_vs_sma200 > 0,
        pd.notna(return_6m) and return_6m > 0,
        pd.notna(return_12m) and return_12m > 0,
    ]
    return int(sum(checks))


def cash_generation_score(latest_fcf, fcf_margin, cash_conversion, capex_intensity,
                           shareholder_yield, fcf_series):
    '''0-6 score of cash-generation quality and capital discipline.'''
    fcf_growing = fcf_series.iloc[-1] >= fcf_series.iloc[0] if len(fcf_series) >= 2 else False
    checks = [
        pd.notna(latest_fcf) and latest_fcf > 0,
        pd.notna(fcf_margin) and fcf_margin > min_fcf_margin,
        pd.notna(cash_conversion) and cash_conversion > min_cash_conversion,
        pd.notna(capex_intensity) and capex_intensity < max_capex_intensity,
        pd.notna(shareholder_yield) and shareholder_yield > min_shareholder_yield,
        fcf_growing,
    ]
    return int(sum(checks))


def leverage_score(debt_to_equity, net_debt_to_ebitda, interest_coverage,
                    current_ratio, quick_ratio, net_debt_change):
    '''0-6, HIGHER = SAFER (opposite direction to raw leverage). Net debt
    falling year-on-year (or already net cash) counts as deleveraging.'''
    checks = [
        pd.notna(debt_to_equity) and debt_to_equity < leverage_debt_to_equity_max,
        pd.notna(net_debt_to_ebitda) and net_debt_to_ebitda < leverage_net_debt_to_ebitda_max,
        pd.notna(interest_coverage) and interest_coverage > leverage_interest_coverage_min,
        pd.notna(current_ratio) and current_ratio > leverage_current_ratio_min,
        pd.notna(quick_ratio) and quick_ratio > leverage_quick_ratio_min,
        pd.notna(net_debt_change) and net_debt_change <= 0,
    ]
    return int(sum(checks))


def dcf_intrinsic_value(latest_fcf, fcf_cagr, net_debt, shares_outstanding):
    '''Bare-bones 5-year Free Cash Flow DCF: grow FCF at its own (clamped)
    historical CAGR for `dcf_projection_years`, discount at `dcf_discount_rate`,
    add a Gordon-growth terminal value, subtract net debt, divide by shares.
    Returns (value_per_share, growth_rate_used) - both NaN if the inputs can't
    support a sane valuation (no FCF, no shares outstanding, discount rate not
    above terminal growth).'''
    if pd.isna(latest_fcf) or latest_fcf <= 0 or pd.isna(shares_outstanding) or shares_outstanding <= 0:
        return np.nan, np.nan
    if dcf_discount_rate <= dcf_terminal_growth:
        return np.nan, np.nan

    growth_rate = fcf_cagr if pd.notna(fcf_cagr) else dcf_growth_fallback
    growth_rate = float(np.clip(growth_rate, dcf_growth_floor, dcf_growth_cap))

    pv_sum = 0.0
    projected_fcf = latest_fcf
    for year in range(1, dcf_projection_years + 1):
        projected_fcf = projected_fcf * (1 + growth_rate)
        pv_sum += projected_fcf / (1 + dcf_discount_rate) ** year

    terminal_value = projected_fcf * (1 + dcf_terminal_growth) / (dcf_discount_rate - dcf_terminal_growth)
    pv_terminal = terminal_value / (1 + dcf_discount_rate) ** dcf_projection_years

    enterprise_value = pv_sum + pv_terminal
    net_debt_for_calc = net_debt if pd.notna(net_debt) else 0.0
    equity_value = enterprise_value - net_debt_for_calc
    value_per_share = equity_value / shares_outstanding
    return value_per_share, growth_rate

In [ ]:
def fetch_asx200_constituents():
    '''Scrapes the current S&P/ASX 200 constituent list (ticker code, company,
    sector) from Wikipedia. Index membership is rebalanced quarterly, so this
    is intentionally fetched live rather than frozen in the notebook - it will
    reflect whatever Wikipedia has at run time. If Wikipedia's page layout has
    changed, this raises a clear error rather than silently screening the
    wrong (or an empty) universe.'''
    url = 'https://en.wikipedia.org/wiki/S%26P/ASX_200'
    try:
        resp = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=20)
        resp.raise_for_status()
        tables = pd.read_html(io.StringIO(resp.text))
    except Exception as exc:
        raise RuntimeError(
            f'Could not fetch the ASX 200 constituent list from Wikipedia ({exc!r}). '
            "Check your internet connection/proxy settings, or set universe_mode = "
            "'watchlist' above and re-run while this is unavailable."
        ) from exc

    constituents = None
    for table in tables:
        cols = [str(c).strip() for c in table.columns]
        if 'Code' in cols and 'Company' in cols:
            constituents = table.rename(columns=lambda c: str(c).strip())
            break

    if constituents is None:
        raise RuntimeError(
            "Fetched the Wikipedia page but couldn't find a table with 'Code' and "
            "'Company' columns - the page layout may have changed. Inspect "
            "https://en.wikipedia.org/wiki/S%26P/ASX_200 manually, or set "
            "universe_mode = 'watchlist' above."
        )

    constituents['Ticker'] = constituents['Code'].astype(str).str.strip().str.upper() + '.AX'
    keep_cols = ['Ticker', 'Company'] + (['Sector'] if 'Sector' in constituents.columns else [])
    result = constituents[keep_cols].dropna(subset=['Ticker'])
    result = result.drop_duplicates(subset='Ticker').reset_index(drop=True)
    if result.empty:
        raise RuntimeError('Parsed the ASX 200 table but ended up with zero tickers - inspect wiki_table manually.')
    return result


sector_lookup = {}
if universe_mode == 'asx200':
    asx200_df = fetch_asx200_constituents()
    tickers = asx200_df['Ticker'].tolist()
    if 'Sector' in asx200_df.columns:
        sector_lookup = dict(zip(asx200_df['Ticker'], asx200_df['Sector']))
    universe_label = 'ASX 200'
elif universe_mode == 'watchlist':
    if active_watchlist not in watchlists:
        raise KeyError(f"active_watchlist '{active_watchlist}' is not in watchlists: {list(watchlists)}")
    tickers = watchlists[active_watchlist]
    universe_label = active_watchlist
else:  # 'single'
    tickers = [single_ticker]
    universe_label = single_ticker.replace('.AX', '')

print(f'Universe: {universe_label} ({len(tickers)} ticker(s))')
if universe_mode == 'asx200':
    print('Full ASX 200 scan - this can take well over 10 minutes given yfinance rate limits.')

In [ ]:
# auto_adjust=True means these are total-return (dividend + split adjusted)
# prices, which is the correct series to use for the price-performance chart.
def to_naive_daily(price_series):
    '''Strip tz-awareness, keeping the local trading-day date. yfinance
    returns each exchange's OWN local timezone (Australia/Sydney for ASX
    names, America/New_York for US ones) - combining tz-aware series from
    DIFFERENT timezones into one DataFrame aligns by absolute instant, not by
    calendar day, which silently produces a mostly-NaN column. Stripping tz
    right after download, before anything is combined, avoids that.'''
    if price_series.index.tz is not None:
        price_series = price_series.copy()
        price_series.index = price_series.index.tz_localize(None)
    return price_series


benchmark_prices = {}
for label, bench_ticker in [('ASX 200', asx200_ticker), ('S&P 500', sp500_ticker)]:
    series = yf.Ticker(bench_ticker).history(period=price_period, auto_adjust=True)['Close'].dropna()
    series = to_naive_daily(series)
    if series.empty:
        raise RuntimeError(
            f'No price data returned for benchmark {bench_ticker} ({label}). '
            'Check your internet connection/proxy settings before continuing - '
            'nothing downstream can run without the benchmark series.'
        )
    benchmark_prices[label] = series
    time.sleep(request_pause_seconds)

print('Benchmarks loaded:', {k: f'{v.index[0].date()} to {v.index[-1].date()}' for k, v in benchmark_prices.items()})

In [ ]:
records = []
trend_frames = []
price_series = {}
errors = []

for i, ticker in enumerate(tickers, 1):
    print(f'[{i}/{len(tickers)}] {ticker}')
    try:
        company = yf.Ticker(ticker)
        info = safe_info(company)

        # --- Price / momentum ------------------------------------------------
        prices = company.history(period=price_period, auto_adjust=True)['Close'].dropna()
        prices = to_naive_daily(prices)
        if prices.empty:
            errors.append({'Ticker': ticker, 'Issue': 'No price history returned (check the ticker is correct and still listed).'})
            time.sleep(request_pause_seconds)
            continue
        price_series[ticker] = prices

        current_price = prices.iloc[-1]
        sma200 = prices.rolling(200).mean().iloc[-1] if len(prices) >= 200 else np.nan
        pct_vs_sma200 = (current_price / sma200 - 1) if pd.notna(sma200) and sma200 != 0 else np.nan

        return_6m = (current_price / prices.iloc[-126] - 1) if len(prices) > 126 else np.nan
        return_12m = (current_price / prices.iloc[-252] - 1) if len(prices) > 252 else np.nan

        # --- Fundamentals ------------------------------------------------------
        shares_outstanding = info.get('sharesOutstanding')
        market_cap = info.get('marketCap')
        if market_cap is None and current_price and shares_outstanding:
            market_cap = current_price * shares_outstanding

        balance_sheet, bs_source = get_financials(company, 'balance_sheet', 'quarterly_balance_sheet')
        income_statement, is_source = get_financials(company, 'income_stmt', 'quarterly_income_stmt')
        cash_flow, cf_source = get_financials(company, 'cashflow', 'quarterly_cashflow')
        fundamentals_note = f'BS:{bs_source}, IS:{is_source}, CF:{cf_source}'

        if balance_sheet.empty and income_statement.empty and cash_flow.empty:
            errors.append({
                'Ticker': ticker,
                'Issue': (
                    'Yahoo Finance returned no annual or quarterly financial statements - '
                    'Graham/Cash Generation/Leverage/DCF metrics could not be calculated. '
                    'Momentum (price-based) metrics are still valid below.'
                )
            })

        current_assets = safe_get(balance_sheet, ['Current Assets'])
        current_liabilities = safe_get(balance_sheet, ['Current Liabilities'])
        cash = safe_get(balance_sheet, [
            'Cash And Cash Equivalents', 'Cash Cash Equivalents And Short Term Investments', 'Cash'
        ])
        receivables = safe_get(balance_sheet, ['Accounts Receivable', 'Receivables'])
        short_term_investments = safe_get(balance_sheet, [
            'Other Short Term Investments', 'Cash Cash Equivalents And Short Term Investments'
        ])
        total_debt = safe_get(balance_sheet, ['Total Debt'])
        shareholder_equity = safe_get(balance_sheet, ['Stockholders Equity', 'Total Equity Gross Minority Interest'])

        revenue = safe_get(income_statement, ['Total Revenue'])
        operating_income = safe_get(income_statement, ['Operating Income'])
        net_income = safe_get(income_statement, ['Net Income'])
        ebit = safe_get(income_statement, ['EBIT', 'Operating Income'])
        ebitda = safe_get(income_statement, ['EBITDA', 'Normalized EBITDA'])
        interest_expense = abs(safe_get(income_statement, ['Interest Expense']))
        depreciation_amortisation = safe_get(cash_flow, ['Depreciation And Amortization', 'Depreciation Amortization Depletion'])
        if pd.isna(ebitda) and pd.notna(ebit) and pd.notna(depreciation_amortisation):
            ebitda = ebit + depreciation_amortisation

        current_ratio = current_assets / current_liabilities if pd.notna(current_assets) and pd.notna(current_liabilities) and current_liabilities != 0 else np.nan
        quick_assets = np.nansum([cash, receivables]) if not (pd.isna(cash) and pd.isna(receivables)) else np.nan
        if pd.notna(quick_assets) and pd.notna(short_term_investments):
            quick_assets += short_term_investments
        quick_ratio = quick_assets / current_liabilities if pd.notna(quick_assets) and pd.notna(current_liabilities) and current_liabilities != 0 else np.nan

        debt_to_equity = total_debt / shareholder_equity if pd.notna(total_debt) and pd.notna(shareholder_equity) and shareholder_equity != 0 else np.nan
        net_debt = (total_debt - cash) if pd.notna(total_debt) and pd.notna(cash) else np.nan
        net_debt_to_ebitda = net_debt / ebitda if pd.notna(net_debt) and pd.notna(ebitda) and ebitda != 0 else np.nan
        interest_coverage = ebit / interest_expense if pd.notna(ebit) and pd.notna(interest_expense) and interest_expense != 0 else np.nan

        roe = net_income / shareholder_equity if pd.notna(net_income) and pd.notna(shareholder_equity) and shareholder_equity != 0 else np.nan
        operating_margin = operating_income / revenue if pd.notna(operating_income) and pd.notna(revenue) and revenue != 0 else np.nan

        # --- Graham -------------------------------------------------------------
        eps = info.get('trailingEps')
        bvps = info.get('bookValue')
        pe = info.get('trailingPE')
        pb = info.get('priceToBook')
        try:
            pe = float(pe) if pe is not None else np.nan
        except (TypeError, ValueError):
            pe = np.nan
        try:
            pb = float(pb) if pb is not None else np.nan
        except (TypeError, ValueError):
            pb = np.nan
        if pd.isna(pb) and bvps and current_price:
            pb = current_price / bvps

        graham_number = np.sqrt(22.5 * eps * bvps) if eps and bvps and eps > 0 and bvps > 0 else np.nan
        margin_of_safety = (graham_number / current_price - 1) if pd.notna(graham_number) and current_price else np.nan

        g_score = graham_score(eps, pe, pb, current_ratio, debt_to_equity, margin_of_safety)
        m_score = momentum_score(pct_vs_sma200, return_6m, return_12m)

        # --- Cash generation / trend / DCF --------------------------------------
        revenue_series = get_row_series(income_statement, ['Total Revenue'])
        net_income_series = get_row_series(income_statement, ['Net Income'])
        ocf_series = get_row_series(cash_flow, ['Operating Cash Flow'])
        capex_series = get_row_series(cash_flow, ['Capital Expenditure']).abs()
        dividends_series = get_row_series(cash_flow, ['Cash Dividends Paid', 'Common Stock Dividend Paid']).abs()
        buybacks_series = get_row_series(cash_flow, ['Repurchase Of Capital Stock', 'Common Stock Repurchased']).abs()
        debt_issued_series = get_row_series(cash_flow, ['Long Term Debt Issuance'])
        debt_repaid_series = get_row_series(cash_flow, ['Long Term Debt Payments', 'Repayment Of Debt']).abs()
        cash_balance_series = get_row_series(balance_sheet, [
            'Cash And Cash Equivalents', 'Cash Cash Equivalents And Short Term Investments', 'Cash'
        ])
        total_debt_series = get_row_series(balance_sheet, ['Total Debt'])

        fcf_series = (ocf_series - capex_series).dropna()
        operating_cash_flow = ocf_series.iloc[-1] if not ocf_series.empty else np.nan
        capex = capex_series.iloc[-1] if not capex_series.empty else np.nan
        latest_fcf = fcf_series.iloc[-1] if not fcf_series.empty else np.nan
        latest_dividends = dividends_series.iloc[-1] if not dividends_series.empty else np.nan
        latest_buybacks = buybacks_series.iloc[-1] if not buybacks_series.empty else np.nan

        fcf_margin = latest_fcf / revenue if pd.notna(latest_fcf) and pd.notna(revenue) and revenue else np.nan
        cash_conversion = operating_cash_flow / net_income if pd.notna(operating_cash_flow) and pd.notna(net_income) and net_income else np.nan
        capex_intensity = capex / revenue if pd.notna(capex) and pd.notna(revenue) and revenue else np.nan
        cash_returned = np.nansum([latest_dividends, latest_buybacks]) if not (pd.isna(latest_dividends) and pd.isna(latest_buybacks)) else np.nan
        shareholder_yield = cash_returned / market_cap if pd.notna(cash_returned) and market_cap else np.nan
        fcf_yield = latest_fcf / market_cap if pd.notna(latest_fcf) and market_cap else np.nan

        years_span = (fcf_series.index[-1] - fcf_series.index[0]).days / 365.25 if len(fcf_series) >= 2 else np.nan
        fcf_cagr = calc_cagr(fcf_series.iloc[0], fcf_series.iloc[-1], years_span) if pd.notna(years_span) else np.nan

        cg_score = cash_generation_score(latest_fcf, fcf_margin, cash_conversion, capex_intensity, shareholder_yield, fcf_series)

        dcf_value_per_share, dcf_growth_used = dcf_intrinsic_value(latest_fcf, fcf_cagr, net_debt, shares_outstanding)
        dcf_upside_pct = (dcf_value_per_share / current_price - 1) if pd.notna(dcf_value_per_share) and current_price else np.nan

        # Net debt trend: latest period's net debt vs the earliest available in
        # the multi-period window. Falls back to debt-repaid > debt-issued in
        # the latest period if there isn't enough balance-sheet history.
        net_debt_change = np.nan
        if len(total_debt_series) >= 2 and len(cash_balance_series) >= 2:
            net_debt_hist = (total_debt_series - cash_balance_series).dropna()
            if len(net_debt_hist) >= 2:
                net_debt_change = net_debt_hist.iloc[-1] - net_debt_hist.iloc[0]
        if pd.isna(net_debt_change) and not debt_repaid_series.empty and not debt_issued_series.empty:
            net_debt_change = debt_issued_series.iloc[-1] - debt_repaid_series.iloc[-1]

        l_score = leverage_score(debt_to_equity, net_debt_to_ebitda, interest_coverage, current_ratio, quick_ratio, net_debt_change)

        composite_score = g_score + m_score + cg_score + l_score

        # --- Trend table (for the Cash_Flow_Trends sheet / FCF chart) ----------
        trend = pd.DataFrame({
            'Revenue': revenue_series,
            'Net Income': net_income_series,
            'Operating Cash Flow': ocf_series,
            'CapEx': capex_series,
            'Free Cash Flow': fcf_series,
            'Dividends Paid': dividends_series,
            'Buybacks': buybacks_series,
            'Total Debt': total_debt_series,
            'Cash & Equivalents': cash_balance_series,
        })
        if not trend.empty:
            trend.insert(0, 'Ticker', ticker.replace('.AX', ''))
            trend.index.name = 'Period End'
            trend_frames.append(trend.reset_index())

        records.append({
            'Ticker': ticker.replace('.AX', ''),
            'Company': info.get('shortName', ticker),
            'Sector': sector_lookup.get(ticker, info.get('sector')),
            'Price': current_price,
            'Market Cap': market_cap,

            # Graham
            'EPS': eps,
            'Book Value/Share': bvps,
            'P/E': pe,
            'P/B': pb,
            'Graham Number': graham_number,
            'Margin of Safety %': margin_of_safety,
            'Graham Score (0-6)': g_score,

            # Momentum
            'SMA200': sma200,
            '% vs SMA200': pct_vs_sma200,
            '6M Return': return_6m,
            '12M Return': return_12m,
            'Momentum Score (0-3)': m_score,

            # Cash generation / DCF
            'Revenue': revenue,
            'Operating Cash Flow': operating_cash_flow,
            'CapEx': capex,
            'Free Cash Flow': latest_fcf,
            'FCF Margin': fcf_margin,
            'Cash Conversion (OCF/NI)': cash_conversion,
            'CapEx Intensity': capex_intensity,
            'Shareholder Yield': shareholder_yield,
            'FCF Yield': fcf_yield,
            'FCF CAGR': fcf_cagr,
            'Cash Generation Score (0-6)': cg_score,
            'DCF Value/Share': dcf_value_per_share,
            'DCF Upside %': dcf_upside_pct,
            'DCF Growth Rate Used': dcf_growth_used,

            # Leverage
            'Current Ratio': current_ratio,
            'Quick Ratio': quick_ratio,
            'Debt/Equity': debt_to_equity,
            'Net Debt': net_debt,
            'EBITDA': ebitda,
            'Net Debt/EBITDA': net_debt_to_ebitda,
            'Interest Coverage': interest_coverage,
            'Net Debt Change (period)': net_debt_change,
            'Leverage Score (0-6)': l_score,

            # Context
            'ROE': roe,
            'Operating Margin': operating_margin,
            'Composite Score (0-24)': composite_score,
            'Fundamentals Source': fundamentals_note,
        })

    except Exception as exc:
        errors.append({'Ticker': ticker, 'Issue': str(exc)})

    time.sleep(request_pause_seconds)

In [ ]:
summary = pd.DataFrame(records)
errors_df = pd.DataFrame(errors)
trend_all = pd.concat(trend_frames, ignore_index=True) if trend_frames else pd.DataFrame()

if summary.empty:
    raise RuntimeError(
        'No stocks were successfully analysed. Check the ticker list, your internet access, '
        'and the errors captured in `errors_df`:\n'
        f'{errors_df.to_string(index=False) if not errors_df.empty else "(no per-ticker errors were recorded either - check connectivity)"}'
    )

if not errors_df.empty:
    print(f'Warning: {len(errors_df)} of {len(tickers)} tickers could not be fully analysed - see errors_df / the Data_Issues sheet.')

summary = summary.sort_values(by='Composite Score (0-24)', ascending=False).reset_index(drop=True)

graham_sheet = summary[[
    'Ticker', 'Company', 'Sector', 'Price', 'EPS', 'Book Value/Share', 'P/E', 'P/B',
    'Current Ratio', 'Debt/Equity', 'Graham Number', 'Margin of Safety %', 'Graham Score (0-6)'
]].sort_values(by=['Graham Score (0-6)', 'Margin of Safety %'], ascending=[False, False]).reset_index(drop=True)

momentum_sheet = summary[[
    'Ticker', 'Company', 'Sector', 'Price', 'SMA200', '% vs SMA200', '6M Return', '12M Return', 'Momentum Score (0-3)'
]].sort_values(by=['Momentum Score (0-3)', '% vs SMA200'], ascending=[False, False]).reset_index(drop=True)

cashgen_sheet = summary[[
    'Ticker', 'Company', 'Sector', 'Market Cap', 'Revenue', 'Operating Cash Flow', 'CapEx', 'Free Cash Flow',
    'FCF Margin', 'Cash Conversion (OCF/NI)', 'CapEx Intensity', 'Shareholder Yield', 'FCF Yield', 'FCF CAGR',
    'Cash Generation Score (0-6)'
]].sort_values(by=['Cash Generation Score (0-6)', 'FCF Yield'], ascending=[False, False]).reset_index(drop=True)

dcf_sheet = summary[[
    'Ticker', 'Company', 'Sector', 'Price', 'Free Cash Flow', 'FCF CAGR', 'DCF Growth Rate Used',
    'DCF Value/Share', 'DCF Upside %'
]].sort_values(by='DCF Upside %', ascending=False).reset_index(drop=True)

leverage_sheet = summary[[
    'Ticker', 'Company', 'Sector', 'Current Ratio', 'Quick Ratio', 'Debt/Equity', 'Net Debt',
    'EBITDA', 'Net Debt/EBITDA', 'Interest Coverage', 'Net Debt Change (period)', 'Leverage Score (0-6)'
]].sort_values(by='Leverage Score (0-6)', ascending=False).reset_index(drop=True)

candidates = summary[
    (summary['Graham Score (0-6)'] >= min_graham_score) &
    (summary['Momentum Score (0-3)'] >= min_momentum_score) &
    (summary['Cash Generation Score (0-6)'] >= min_cashgen_score) &
    (summary['Leverage Score (0-6)'] >= min_leverage_score)
].sort_values(by='Composite Score (0-24)', ascending=False).reset_index(drop=True)

print(f'{len(candidates)} of {len(summary)} analysed tickers clear all four minimum scores.')

assumptions = pd.DataFrame({
    'Input': [
        'Universe Mode', 'Universe', 'Tickers Analysed', 'Price Period',
        'Graham: max P/E', 'Graham: max P/B', 'Graham: min Current Ratio', 'Graham: max Debt/Equity',
        'Cash Gen: min FCF Margin', 'Cash Gen: min Cash Conversion', 'Cash Gen: max CapEx Intensity', 'Cash Gen: min Shareholder Yield',
        'DCF: Discount Rate', 'DCF: Terminal Growth', 'DCF: Projection Years', 'DCF: Growth Floor/Cap',
        'Leverage: max Debt/Equity', 'Leverage: max Net Debt/EBITDA', 'Leverage: min Interest Coverage',
        'Leverage: min Current Ratio', 'Leverage: min Quick Ratio',
        'Candidates: min Graham/Momentum/CashGen/Leverage Score',
        'Generated at',
    ],
    'Value': [
        universe_mode, universe_label, len(summary), price_period,
        graham_pe_max, graham_pb_max, graham_current_ratio_min, graham_debt_to_equity_max,
        min_fcf_margin, min_cash_conversion, max_capex_intensity, min_shareholder_yield,
        dcf_discount_rate, dcf_terminal_growth, dcf_projection_years, f'{dcf_growth_floor} / {dcf_growth_cap}',
        leverage_debt_to_equity_max, leverage_net_debt_to_ebitda_max, leverage_interest_coverage_min,
        leverage_current_ratio_min, leverage_quick_ratio_min,
        f'{min_graham_score}/{min_momentum_score}/{min_cashgen_score}/{min_leverage_score}',
        datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    ]
})

In [ ]:
# --- Normalised price chart data (top N by Composite Score + both benchmarks) --
chart_tickers = summary.sort_values('Composite Score (0-24)', ascending=False)['Ticker'].head(chart_max_tickers).tolist()
chart_price_map = {t: price_series[t + '.AX'] for t in chart_tickers if (t + '.AX') in price_series}

normalised_prices = pd.DataFrame({
    ticker: (series / series.iloc[0]) * 100
    for ticker, series in chart_price_map.items()
})
for label, series in benchmark_prices.items():
    normalised_prices[label] = (series / series.iloc[0]) * 100
normalised_prices.index.name = 'Date'

# Excel can't store timezone-aware datetimes - strip tz if yfinance returned one.
if normalised_prices.index.tz is not None:
    normalised_prices.index = normalised_prices.index.tz_localize(None)
if not trend_all.empty and pd.api.types.is_datetime64tz_dtype(trend_all['Period End']):
    trend_all['Period End'] = trend_all['Period End'].dt.tz_localize(None)

In [ ]:
with pd.ExcelWriter(output_file, engine='xlsxwriter', datetime_format='yyyy-mm-dd') as writer:
    workbook = writer.book

    title_fmt = workbook.add_format({
        'bold': True, 'font_size': 14, 'font_color': '#FFFFFF',
        'bg_color': '#0B4F6C', 'align': 'left', 'valign': 'vcenter'
    })
    header_fmt = workbook.add_format({
        'bold': True, 'font_color': '#FFFFFF', 'bg_color': '#1F6F8B',
        'border': 0, 'align': 'center', 'valign': 'vcenter', 'text_wrap': True
    })
    percent_fmt = workbook.add_format({'num_format': '0.0%'})
    ratio_fmt = workbook.add_format({'num_format': '0.00'})
    multiple_fmt = workbook.add_format({'num_format': '0.00"x"'})
    number_fmt = workbook.add_format({'num_format': '#,##0'})
    date_fmt = workbook.add_format({'num_format': 'yyyy-mm-dd'})
    text_fmt = workbook.add_format({})

    # Column width/format lookup shared by every metrics sheet below. Looked up
    # by COLUMN NAME (not hardcoded letter ranges) so it stays correct even if
    # columns are added, removed or reordered later.
    column_format_map = {
        'Ticker': (10, text_fmt), 'Company': (26, text_fmt), 'Sector': (22, text_fmt),
        'Price': (10, ratio_fmt), 'Market Cap': (16, number_fmt),
        'EPS': (10, ratio_fmt), 'Book Value/Share': (14, ratio_fmt), 'P/E': (10, ratio_fmt), 'P/B': (10, ratio_fmt),
        'Graham Number': (14, ratio_fmt), 'Margin of Safety %': (16, percent_fmt), 'Graham Score (0-6)': (14, ratio_fmt),
        'SMA200': (10, ratio_fmt), '% vs SMA200': (14, percent_fmt), '6M Return': (12, percent_fmt),
        '12M Return': (12, percent_fmt), 'Momentum Score (0-3)': (16, ratio_fmt),
        'Revenue': (16, number_fmt), 'Operating Cash Flow': (18, number_fmt), 'CapEx': (14, number_fmt),
        'Free Cash Flow': (16, number_fmt), 'FCF Margin': (12, percent_fmt),
        'Cash Conversion (OCF/NI)': (18, ratio_fmt), 'CapEx Intensity': (14, percent_fmt),
        'Shareholder Yield': (16, percent_fmt), 'FCF Yield': (12, percent_fmt), 'FCF CAGR': (12, percent_fmt),
        'Cash Generation Score (0-6)': (18, ratio_fmt),
        'DCF Growth Rate Used': (16, percent_fmt), 'DCF Value/Share': (14, ratio_fmt), 'DCF Upside %': (14, percent_fmt),
        'Current Ratio': (14, ratio_fmt), 'Quick Ratio': (14, ratio_fmt), 'Debt/Equity': (14, ratio_fmt),
        'Net Debt': (16, number_fmt), 'EBITDA': (16, number_fmt), 'Net Debt/EBITDA': (16, multiple_fmt),
        'Interest Coverage': (16, multiple_fmt), 'Net Debt Change (period)': (18, number_fmt),
        'Leverage Score (0-6)': (16, ratio_fmt),
        'ROE': (10, percent_fmt), 'Operating Margin': (14, percent_fmt), 'Composite Score (0-24)': (18, ratio_fmt),
        'Fundamentals Source': (26, text_fmt),
    }

    def write_metrics_sheet(sheet_name, frame, title, score_column=None):
        frame.to_excel(writer, sheet_name=sheet_name, index=False, startrow=2)
        ws = writer.sheets[sheet_name]
        last_col_letter = excel_column_name(len(frame.columns) - 1)
        ws.merge_range(f'A1:{last_col_letter}1', title, title_fmt)
        ws.freeze_panes(3, 0)
        ws.set_row(2, 34)
        ws.autofilter(2, 0, len(frame) + 2, len(frame.columns) - 1)
        for col_num, column in enumerate(frame.columns):
            width, fmt = column_format_map.get(column, (16, ratio_fmt))
            ws.set_column(col_num, col_num, width, fmt)
            ws.write(2, col_num, column, header_fmt)
        if score_column and score_column in frame.columns:
            score_col = frame.columns.get_loc(score_column)
            ws.conditional_format(3, score_col, len(frame) + 2, score_col, {
                'type': '3_color_scale', 'min_color': '#F8696B', 'mid_color': '#FFEB84', 'max_color': '#63BE7B'
            })
        return ws

    write_metrics_sheet('Summary', summary, f'ASX Multi-Factor Screener: {universe_label}', 'Composite Score (0-24)')
    write_metrics_sheet('Candidates', candidates, f'Candidates Passing All Four Minimum Scores: {universe_label}', 'Composite Score (0-24)')
    write_metrics_sheet('Graham', graham_sheet, 'Graham Value Baseline', 'Graham Score (0-6)')
    write_metrics_sheet('Momentum', momentum_sheet, '200-Day SMA Momentum', 'Momentum Score (0-3)')
    write_metrics_sheet('Cash_Generation', cashgen_sheet, 'Cash Generation & Free Cash Flow', 'Cash Generation Score (0-6)')
    write_metrics_sheet('DCF', dcf_sheet, 'Basic Free Cash Flow DCF', None)
    write_metrics_sheet('Leverage', leverage_sheet, 'Leverage & Balance-Sheet Risk (higher score = safer)', 'Leverage Score (0-6)')

    # --- Price / trend / assumptions / errors sheets ---------------------------
    normalised_prices.to_excel(writer, sheet_name='Price_Normalised')
    ws_prices = writer.sheets['Price_Normalised']
    ws_prices.freeze_panes(1, 1)
    ws_prices.set_column('A:A', 13, date_fmt)
    ws_prices.set_column(1, len(normalised_prices.columns), 14, ratio_fmt)
    for col_num, column in enumerate(['Date'] + normalised_prices.columns.tolist()):
        ws_prices.write(0, col_num, column, header_fmt)

    for sheet_name, frame, date_first_col in [
        ('Cash_Flow_Trends', trend_all, True),
        ('Assumptions', assumptions, False),
        ('Data_Issues', errors_df, False),
    ]:
        frame.to_excel(writer, sheet_name=sheet_name, index=False)
        ws = writer.sheets[sheet_name]
        ws.freeze_panes(1, 0)
        ws.set_column(0, max(len(frame.columns) - 1, 0), 22)
        if date_first_col and not frame.empty:
            ws.set_column(1, 1, 13, date_fmt)
        for col_num, column in enumerate(frame.columns):
            ws.write(0, col_num, column, header_fmt)

    # --- Charts sheet ------------------------------------------------------------
    ws_charts = workbook.add_worksheet('Charts')
    ws_charts.merge_range('A1:L1', f'ASX Multi-Factor Screener Charts: {universe_label}', title_fmt)
    ws_charts.set_column('A:L', 13)

    performance_chart = workbook.add_chart({'type': 'line'})
    performance_chart.set_title({'name': f'Normalised Price vs ASX 200 / S&P 500 (Base = 100, top {len(chart_tickers)} by Composite Score)'})
    performance_chart.set_x_axis({'name': 'Date', 'date_axis': True, 'num_format': 'yyyy'})
    performance_chart.set_y_axis({'name': 'Indexed Price'})
    performance_chart.set_legend({'position': 'bottom'})
    performance_chart.set_size({'width': 920, 'height': 420})
    last_row_prices = len(normalised_prices)
    for col_index, name in enumerate(normalised_prices.columns, start=1):
        excel_col = excel_column_name(col_index)
        is_benchmark = name in ('ASX 200', 'S&P 500')
        performance_chart.add_series({
            'name': name,
            'categories': f'=Price_Normalised!$A$2:$A${last_row_prices + 1}',
            'values': f'=Price_Normalised!${excel_col}$2:${excel_col}${last_row_prices + 1}',
            'line': {'width': 2.5 if is_benchmark else 1.0, 'dash_type': 'dash' if is_benchmark else 'solid'}
        })
    ws_charts.insert_chart('A3', performance_chart)

    if len(summary) > 1:
        top_scores = summary.sort_values('Composite Score (0-24)', ascending=False).head(chart_max_tickers)
        score_chart = workbook.add_chart({'type': 'column'})
        score_chart.set_title({'name': f'Composite Score by Ticker (top {len(top_scores)})'})
        # Wired off a helper sheet so the chart has a stable, sorted-and-capped
        # range to point at regardless of how big/unsorted Summary is.
        top_scores[['Ticker', 'Composite Score (0-24)']].to_excel(writer, sheet_name='_ChartData', index=False)
        ws_chart_data = writer.sheets['_ChartData']
        score_chart.add_series({
            'name': 'Composite Score (0-24)',
            'categories': f'=_ChartData!$A$2:$A${len(top_scores) + 1}',
            'values': f'=_ChartData!$B$2:$B${len(top_scores) + 1}',
            'fill': {'color': '#1F6F8B'}
        })
        score_chart.set_y_axis({'name': 'Composite Score (0-24)'})
        score_chart.set_legend({'none': True})
        score_chart.set_size({'width': 920, 'height': 380})
        ws_charts.insert_chart('A24', score_chart)

        graham_col = excel_column_name(graham_sheet.columns.get_loc('Margin of Safety %'))
        momentum_col_lookup = excel_column_name(summary.columns.get_loc('% vs SMA200'))
        value_momentum_chart = workbook.add_chart({'type': 'scatter', 'subtype': 'straight_with_markers'})
        value_momentum_chart.set_title({'name': 'Value vs Momentum: Margin of Safety % vs % above/below 200-day SMA'})
        value_momentum_chart.set_x_axis({'name': 'Margin of Safety %', 'num_format': '0%'})
        value_momentum_chart.set_y_axis({'name': '% vs SMA200', 'num_format': '0%'})
        value_momentum_chart.set_legend({'none': True})
        value_momentum_chart.set_size({'width': 920, 'height': 380})
        mos_col = excel_column_name(summary.columns.get_loc('Margin of Safety %'))
        data_first_row, data_last_row = 4, len(summary) + 3
        value_momentum_chart.add_series({
            'name': 'Stocks',
            'categories': f'=Summary!${mos_col}${data_first_row}:${mos_col}${data_last_row}',
            'values': f'=Summary!${momentum_col_lookup}${data_first_row}:${momentum_col_lookup}${data_last_row}',
            'marker': {'type': 'circle', 'size': 7, 'border': {'color': '#0B4F6C'}, 'fill': {'color': '#1F6F8B'}}
        })
        ws_charts.insert_chart('A45', value_momentum_chart)

        leverage_col = excel_column_name(summary.columns.get_loc('Debt/Equity'))
        cashgen_col = excel_column_name(summary.columns.get_loc('Cash Generation Score (0-6)'))
        risk_quality_chart = workbook.add_chart({'type': 'scatter', 'subtype': 'straight_with_markers'})
        risk_quality_chart.set_title({'name': 'Risk vs Quality: Debt/Equity vs Cash Generation Score'})
        risk_quality_chart.set_x_axis({'name': 'Debt/Equity'})
        risk_quality_chart.set_y_axis({'name': 'Cash Generation Score (0-6)'})
        risk_quality_chart.set_legend({'none': True})
        risk_quality_chart.set_size({'width': 920, 'height': 380})
        risk_quality_chart.add_series({
            'name': 'Stocks',
            'categories': f'=Summary!${leverage_col}${data_first_row}:${leverage_col}${data_last_row}',
            'values': f'=Summary!${cashgen_col}${data_first_row}:${cashgen_col}${data_last_row}',
            'marker': {'type': 'circle', 'size': 7, 'border': {'color': '#009E84'}, 'fill': {'color': '#63BE7B'}}
        })
        ws_charts.insert_chart('A66', risk_quality_chart)

print(f'Excel workbook created successfully:\n{output_file}')

In [ ]:
# --- OPTIONAL DEBUG CELL --------------------------------------------------
# Run this on its own for a single ticker if it's showing blank fundamentals
# in the Summary sheet. It prints exactly what Yahoo Finance returned so you
# can see whether the statement is genuinely empty or just uses different
# row labels than the ones safe_get()/get_row_series() are looking for.
debug_ticker = tickers[0]
debug_company = yf.Ticker(debug_ticker)

print(f'--- Debug: {debug_ticker} ---')
for label, attr in [
    ('Annual balance sheet', 'balance_sheet'),
    ('Quarterly balance sheet', 'quarterly_balance_sheet'),
    ('Annual income statement', 'income_stmt'),
    ('Quarterly income statement', 'quarterly_income_stmt'),
    ('Annual cash flow', 'cashflow'),
    ('Quarterly cash flow', 'quarterly_cashflow'),
]:
    try:
        df = getattr(debug_company, attr)
    except Exception as exc:
        print(f'{label}: raised {exc!r}')
        continue
    if df is None or df.empty:
        print(f'{label}: EMPTY (nothing returned by Yahoo Finance)')
    else:
        print(f'{label}: {df.shape[1]} period(s) available. Row labels:')
        print(list(df.index))
    print()